# Demographic–Behavioural Integration

## Overview

This notebook integrates the constrained representative demographic profile space with the behavioural evidence derived from NTS0611a, NTS0502b, and NTS0409a.

For a selected feasible demographic profile, trip-purpose probabilities are first retrieved using age and gender. Demographic trip-purpose constraints are then applied according to industry applicability and distance travelled to work. Purpose-conditioned temporal distributions are retained from NTS0502b, while travel-mode distributions are adjusted using age and household car availability.

The resulting output is a constrained demographic–behavioural representation containing:

- eight demographic attributes;
- a constrained trip-purpose probability distribution;
- a temporal distribution for each remaining trip purpose; and
- a constrained travel-mode distribution for each remaining trip purpose.

The framework retains complete probability distributions and does not sample a single trip purpose, start time, or travel mode. Profiles from the constrained representative demographic profile space are used as feasible framework inputs and do not represent synthetic individuals.

This notebook covers demographic–behavioural integration only. Behaviour-to-POI mapping, spatial opportunity adjustment, destination resolution, and final framework integration are handled in subsequent stages.

## 1. Import Libraries

In [31]:
import json
import sys

import pandas as pd


pd.set_option(
    "display.max_columns",
    None,
)

## 2. Set Up File Paths

In [32]:
import sys
from pathlib import Path

project_root = Path.cwd().parents[1]
sys.path.insert(0, str(project_root))

representative_profile_space = (
    project_root
    / "output"
    / "demographic_profiles"
    / "constrained_representative_profile_space.parquet"
)

behavioural_evidence_layer = (
    project_root
    / "output"
    / "behavioural_evidence"
    / "nts_behavioural_evidence_layer.json"
)

from src.demographic_behaviour_layers_integration import (
    construct_constrained_behaviour_representation,
)

print("Project root:")
print(project_root)

print("\nPopulation input:")
print(representative_profile_space)

print("\nBehavioural evidence input:")
print(behavioural_evidence_layer)

print("\nConstraint functions imported successfully.")

Project root:
c:\Users\user\OneDrive - University of Stirling\Desktop\dissertation\mobility-modelling-framework

Population input:
c:\Users\user\OneDrive - University of Stirling\Desktop\dissertation\mobility-modelling-framework\output\demographic_profiles\constrained_representative_profile_space.parquet

Behavioural evidence input:
c:\Users\user\OneDrive - University of Stirling\Desktop\dissertation\mobility-modelling-framework\output\behavioural_evidence\nts_behavioural_evidence_layer.json

Constraint functions imported successfully.


## 3. Load Input Data

In [33]:
population = pd.read_parquet(
    representative_profile_space
)

with open(
    behavioural_evidence_layer,
    "r",
    encoding="utf-8",
) as file:
    behavioural_evidence = json.load(file)

print(
    "Population loaded:",
    population.shape,
)

print(
    "Trip purposes loaded:",
    len(behavioural_evidence),
)

Population loaded: (2267232, 14)
Trip purposes loaded: 14


## 4. Inspect Input Data

The input datasets are inspected before integration to confirm that the constrained representative profile space contains the eight required demographic attributes and that the behavioural evidence contains the fourteen shared trip purposes and their three probability components.

In [34]:
profile_columns = [
    "Age",
    "Gender",
    "Household Type",
    "Economic Activity Status",
    "Industry",
    "Ethnic Group",
    "Distance travelled to work",
    "Car Ownership",
]

missing_columns = [
    column
    for column in profile_columns
    if column not in population.columns
]

if missing_columns:
    raise ValueError(
        "Required demographic columns "
        f"are missing: {missing_columns}"
    )

if len(behavioural_evidence) != 14:
    raise ValueError(
        "Expected 14 shared trip purposes, "
        f"but found {len(behavioural_evidence)}."
    )

display(
    population[
        profile_columns
    ].head()
)

print("Demographic attributes:")
for column in profile_columns:
    print(f"- {column}")

print("\nShared trip purposes:")
for purpose in behavioural_evidence:
    print(f"- {purpose}")

example_purpose = next(
    iter(behavioural_evidence)
)

print(
    "\nExample purpose:",
    example_purpose,
)

print("Behavioural components:")
for component in (
    behavioural_evidence[
        example_purpose
    ]
):
    print(f"- {component}")

,Age,Gender,Household Type,Economic Activity Status,Industry,Ethnic Group,Distance travelled to work,Car Ownership
0,0-4,Female,Couple household: With dependent children,Does not apply,Does not apply,"Asian, Asian British or Asian Welsh: Bangladeshi",Does not apply,1 car or van in household
1,0-4,Female,Couple household: With dependent children,Does not apply,Does not apply,"Asian, Asian British or Asian Welsh: Bangladeshi",Does not apply,2 cars or vans in household
2,0-4,Female,Couple household: With dependent children,Does not apply,Does not apply,"Asian, Asian British or Asian Welsh: Bangladeshi",Does not apply,3 or more cars or vans in household
3,0-4,Female,Couple household: With dependent children,Does not apply,Does not apply,"Asian, Asian British or Asian Welsh: Bangladeshi",Does not apply,No cars or vans in household
4,0-4,Female,Couple household: With dependent children,Does not apply,Does not apply,"Black, Black British, Black Welsh, Caribbean o...",Does not apply,1 car or van in household


Demographic attributes:
- Age
- Gender
- Household Type
- Economic Activity Status
- Industry
- Ethnic Group
- Distance travelled to work
- Car Ownership

Shared trip purposes:
- Commuting
- Business
- Education
- Escort Education
- Shopping
- Other Escort
- Personal Business
- Visiting Friends at Private Home
- Visiting Friends Elsewhere
- Entertainment or Public Activity
- Sport: Participate
- Holiday: Base
- Day Trip
- Just Walk

Example purpose: Commuting
Behavioural components:
- nts0611a_trip_purpose_distribution
- nts0502b_temporal_distribution
- nts0409a_travel_mode_distribution


## 5. Define Integration Function

The integration function accepts one feasible representative demographic profile and constructs its constrained behavioural representation.

Trip-purpose probabilities are retrieved from NTS0611a using age and gender before demographic trip-purpose constraints are applied. For every remaining trip purpose, the corresponding temporal distribution is retrieved from NTS0502b and the travel-mode distribution is retrieved from NTS0409a. Travel-mode constraints are then applied using age and household car availability.

The function retains the complete conditional probability distributions and does not sample individual behavioural outcomes.

## 6. Select a Validation Profile

A feasible profile aged 5–11 from a household without a car or van is selected from the constrained representative demographic profile space. This case activates both the trip-purpose and travel-mode constraints and therefore provides a useful initial integration test.

The profile is selected from the existing profile space rather than manually constructed, ensuring that all eight demographic attributes form a logically feasible combination.

In [35]:
validation_candidates = population[
    (
        population["Age"]
        == "5-11"
    )
    & (
        population["Gender"]
        == "Female"
    )
    & (
        population["Car Ownership"]
        == "No cars or vans in household"
    )
]

if validation_candidates.empty:
    raise ValueError(
        "No matching validation "
        "profile was found."
    )

validation_profile = (
    validation_candidates
    .iloc[0][profile_columns]
)

display(
    validation_profile
    .to_frame(
        name="Profile Value"
    )
)

,Profile Value
Age,5-11
Gender,Female
Household Type,Couple household: With dependent children
Economic Activity Status,Does not apply
Industry,Does not apply
Ethnic Group,White: Other White
Distance travelled to work,Does not apply
Car Ownership,No cars or vans in household


## 7. Construct Constrained Behavioural Representation

The selected demographic profile is passed to the integration function. The resulting representation is inspected to confirm that the relevant trip-purpose and travel-mode constraints have been applied and that all remaining probability distributions are valid.

In [36]:
validation_representation = (
    construct_constrained_behaviour_representation(
        representative_profile=validation_profile,
        assembled_behavioural_evidence=behavioural_evidence,
    )
)

trip_purpose_distribution = (
    validation_representation[
        "behavioural_representation"
    ][
        "trip_purpose_distribution"
    ]
)

purpose_conditioned_distribution = (
    validation_representation[
        "behavioural_representation"
    ][
        "purpose_conditioned_distribution"
    ]
)

assert (
    "Commuting"
    not in trip_purpose_distribution
)

assert (
    "Business"
    not in trip_purpose_distribution
)

for purpose, purpose_distribution in (
    purpose_conditioned_distribution.items()
):
    mode_distribution = (
        purpose_distribution[
            "travel_mode_distribution"
        ]
    )

    assert (
        "Car or van driver"
        not in mode_distribution
    )

    assert (
        "Car or van passenger"
        not in mode_distribution
    )

    assert (
        "Motorcycle"
        not in mode_distribution
    )

    assert (
        "Taxi or minicab"
        in mode_distribution
    )

    assert abs(
        sum(
            purpose_distribution[
                "temporal_distribution"
            ].values()
        )
        - 1
    ) < 1e-8

    assert abs(
        sum(mode_distribution.values())
        - 1
    ) < 1e-8

purpose_summary = pd.DataFrame([
    {
        "Trip Purpose": purpose,
        "Trip Purpose Probability": (
            purpose_distribution[
                "trip_purpose_probability"
            ]
        ),
        "Temporal Interval Count": len(
            purpose_distribution[
                "temporal_distribution"
            ]
        ),
        "Travel Mode Count": len(
            purpose_distribution[
                "travel_mode_distribution"
            ]
        ),
        "Trip Purpose Sum": sum(
            trip_purpose_distribution.values()
        ),
        "Temporal Sum": sum(
            purpose_distribution[
                "temporal_distribution"
            ].values()
        ),
        "Travel Mode Sum": sum(
            purpose_distribution[
                "travel_mode_distribution"
            ].values()
        ),
    }
    for purpose, purpose_distribution
    in purpose_conditioned_distribution.items()
])

display(purpose_summary)

print(
    "Original trip-purpose count:",
    len(behavioural_evidence.keys()),
)

print(
    "Constrained trip-purpose count:",
    len(trip_purpose_distribution),
)

print(
    "Removed trip purposes:",
    sorted(
        set(behavioural_evidence)
        - set(trip_purpose_distribution)
    ),
)

print(
    "\nDemographic–behavioural "
    "integration: PASS"
)

,Trip Purpose,Trip Purpose Probability,Temporal Interval Count,Travel Mode Count,Trip Purpose Sum,Temporal Sum,Travel Mode Sum
0,Education,0.328955,24,8,1.0,1.0,1.0
1,Escort Education,0.044314,24,8,1.0,1.0,1.0
2,Shopping,0.032020,24,8,1.0,1.0,1.0
3,Other Escort,0.142932,24,8,1.0,1.0,1.0
4,Personal Business,0.075901,24,8,1.0,1.0,1.0
5,Visiting Friends at Private Home,0.106983,24,8,1.0,1.0,1.0
6,Visiting Friends Elsewhere,0.043814,24,8,1.0,1.0,1.0
7,Entertainment or Public Activity,0.078524,24,8,1.0,1.0,1.0
8,Sport: Participate,0.036136,24,8,1.0,1.0,1.0
9,Holiday: Base,0.015216,24,8,1.0,1.0,1.0


Original trip-purpose count: 14
Constrained trip-purpose count: 12
Removed trip purposes: ['Business', 'Commuting']

Demographic–behavioural integration: PASS


In [37]:
import inspect

print(
    inspect.signature(
        construct_constrained_behaviour_representation
    )
)

(representative_profile, assembled_behavioural_evidence)


## 8. Select Multi-Profile Validation Cases

Four feasible demographic profiles are selected to cover the principal combinations of trip-purpose and travel-mode constraint conditions:

- a young profile without household car availability;
- an adult who works mainly from home and has household car availability;
- a workplace-based adult without household car availability; and
- a workplace-based adult with household car availability as the control case.

All profiles are selected directly from the constrained representative demographic profile space.

In [38]:
NO_CAR_LABEL = (
    "No cars or vans in household"
)

HAS_CAR_LABEL = (
    "1 car or van in household"
)

WFH_LABEL = (
    "Works mainly from home"
)

ADULT_AGE = "23-34"


def select_validation_profile(
    condition,
    profile_type,
):
    candidates = population[
        condition
    ]

    if candidates.empty:
        raise ValueError(
            "No feasible profile found for "
            f"{profile_type}."
        )

    return (
        candidates
        .iloc[0][profile_columns]
    )


young_no_car_profile = (
    select_validation_profile(
        condition=(
            (
                population["Age"]
                == "5-11"
            )
            & (
                population["Gender"]
                == "Female"
            )
            & (
                population["Car Ownership"]
                == NO_CAR_LABEL
            )
        ),
        profile_type=(
            "Young profile without "
            "household car"
        ),
    )
)


wfh_with_car_profile = (
    select_validation_profile(
        condition=(
            (
                population["Age"]
                == ADULT_AGE
            )
            & (
                population["Industry"]
                != "Does not apply"
            )
            & (
                population[
                    "Distance travelled to work"
                ]
                == WFH_LABEL
            )
            & (
                population["Car Ownership"]
                == HAS_CAR_LABEL
            )
        ),
        profile_type=(
            "Adult working mainly from "
            "home with household car"
        ),
    )
)


workplace_no_car_profile = (
    select_validation_profile(
        condition=(
            (
                population["Age"]
                == ADULT_AGE
            )
            & (
                population["Industry"]
                != "Does not apply"
            )
            & (
                ~population[
                    "Distance travelled to work"
                ].isin({
                    "Does not apply",
                    WFH_LABEL,
                })
            )
            & (
                population["Car Ownership"]
                == NO_CAR_LABEL
            )
        ),
        profile_type=(
            "Workplace-based adult "
            "without household car"
        ),
    )
)


workplace_with_car_profile = (
    select_validation_profile(
        condition=(
            (
                population["Age"]
                == ADULT_AGE
            )
            & (
                population["Industry"]
                != "Does not apply"
            )
            & (
                ~population[
                    "Distance travelled to work"
                ].isin({
                    "Does not apply",
                    WFH_LABEL,
                })
            )
            & (
                population["Car Ownership"]
                == HAS_CAR_LABEL
            )
        ),
        profile_type=(
            "Workplace-based adult "
            "with household car"
        ),
    )
)


validation_cases = {
    "Young, no household car": (
        young_no_car_profile
    ),
    "Adult, work from home, with car": (
        wfh_with_car_profile
    ),
    "Adult, workplace, no household car": (
        workplace_no_car_profile
    ),
    "Adult, workplace, with car": (
        workplace_with_car_profile
    ),
}

validation_profiles_table = pd.DataFrame([
    {
        "Profile Type": profile_type,
        "Age": profile["Age"],
        "Gender": profile["Gender"],
        "Industry": profile["Industry"],
        "Distance travelled to work": (
            profile["Distance travelled to work"]
        ),
        "Car Ownership": profile["Car Ownership"],
    }
    for profile_type, profile
    in validation_cases.items()
])

display(validation_profiles_table)

,Profile Type,Age,Gender,Industry,Distance travelled to work,Car Ownership
0,"Young, no household car",5-11,Female,Does not apply,Does not apply,No cars or vans in household
1,"Adult, work from home, with car",23-34,Female,J Information and communication,Works mainly from home,1 car or van in household
2,"Adult, workplace, no household car",23-34,Female,J Information and communication,10km to less than 30km,No cars or vans in household
3,"Adult, workplace, with car",23-34,Female,J Information and communication,10km to less than 30km,1 car or van in household


## 9. Validate Multi-Profile Integration

Each validation profile is passed through the complete
demographic–behavioural integration function. The validation table is
generated by comparing the original behavioural evidence with the
constrained output. Counts and removed alternatives are therefore derived
programmatically rather than entered manually.

The validation examines trip-purpose, start-time, and travel-mode
distributions, including the number of alternatives before and after
constraint application, the alternatives actually removed, probability
normalisation, and preservation of unaffected distributions.

In [39]:
TOL = 1e-8
META = {"Year", "Purpose", "Trip purpose"}
PURPOSE_KEY = "nts0611a_trip_purpose_distribution"
TIME_KEY = "nts0502b_temporal_distribution"
MODE_KEY = "nts0409a_travel_mode_distribution"


def probabilities(record):
    return {key: float(value) for key, value in record.items() if key not in META}


def normalised(original, constrained):
    total = sum(original[key] for key in constrained)
    return all(abs(value - original[key] / total) < TOL
               for key, value in constrained.items())


def expected_removals(profile):
    purposes = set()
    modes = set()

    if profile["Industry"] == "Does not apply":
        purposes = {"Commuting", "Business"}
    elif profile["Distance travelled to work"] == "Works mainly from home":
        purposes = {"Commuting"}

    if profile["Age"] in {"0-4", "5-11"}:
        modes |= {"Car or van driver", "Motorcycle"}
    if profile["Car Ownership"] == "No cars or vans in household":
        modes |= {"Car or van driver", "Car or van passenger"}

    return purposes, modes


def validate(profile_type, profile):
    result = construct_constrained_behaviour_representation(
        representative_profile=profile,
        assembled_behavioural_evidence=behavioural_evidence,
    )["behavioural_representation"]

    purposes = result["trip_purpose_distribution"]
    conditioned = result["purpose_conditioned_distribution"]
    original_purposes = {}

    for purpose, evidence in behavioural_evidence.items():
        records = [record for record in evidence[PURPOSE_KEY]
                   if record["Age"] == profile["Age"]
                   and record["Gender"] == profile["Gender"]]
        if len(records) != 1:
            raise ValueError(
                f"{profile_type}/{purpose}: expected one purpose record, found {len(records)}"
            )
        original_purposes[purpose] = float(records[0]["Trip Purpose Probability"])

    removed_purposes = set(original_purposes) - set(purposes)
    expected_purposes, expected_modes = expected_removals(profile)

    assert set(purposes) == set(conditioned), f"{profile_type}: purpose keys differ"
    assert removed_purposes == expected_purposes, f"{profile_type}: wrong purposes removed"
    assert not set(purposes) - set(original_purposes), f"{profile_type}: purpose added"
    assert abs(sum(purposes.values()) - 1) < TOL, f"{profile_type}: purposes do not sum to 1"
    assert normalised(original_purposes, purposes), f"{profile_type}: purposes not renormalised"

    removed_modes = []
    time_counts = []
    mode_counts = []

    for purpose, constrained in conditioned.items():
        evidence = behavioural_evidence[purpose]
        if len(evidence[TIME_KEY]) != 1 or len(evidence[MODE_KEY]) != 1:
            raise ValueError(f"{profile_type}/{purpose}: expected one time and one mode record")

        original_time = probabilities(evidence[TIME_KEY][0])
        original_mode = probabilities(evidence[MODE_KEY][0])
        new_time = constrained["temporal_distribution"]
        new_mode = constrained["travel_mode_distribution"]
        removed = set(original_mode) - set(new_mode)

        assert new_time == original_time, f"{profile_type}/{purpose}: time distribution changed"
        assert removed == expected_modes, f"{profile_type}/{purpose}: wrong modes removed"
        assert not set(new_mode) - set(original_mode), f"{profile_type}/{purpose}: mode added"
        assert abs(sum(new_mode.values()) - 1) < TOL, f"{profile_type}/{purpose}: modes do not sum to 1"
        assert normalised(original_mode, new_mode), f"{profile_type}/{purpose}: modes not renormalised"

        removed_modes.append(removed)
        time_counts.append(len(new_time))
        mode_counts.append(len(new_mode))

    assert len({frozenset(x) for x in removed_modes}) == 1, \
        f"{profile_type}: inconsistent mode removal"

    return {
        "Profile Type": profile_type,
        "Age": profile["Age"],
        "Purposes Before": len(original_purposes),
        "Purposes After": len(purposes),
        "Purposes Removed": ", ".join(sorted(removed_purposes)) or "None",
        "Start Times After": min(time_counts) if len(set(time_counts)) == 1 else f"{min(time_counts)}â€“{max(time_counts)}",
        "Modes After": min(mode_counts) if len(set(mode_counts)) == 1 else f"{min(mode_counts)}â€“{max(mode_counts)}",
        "Modes Removed": ", ".join(sorted(expected_modes)) or "None",
        "All Checks Passed": True,
    }


validation_summary_df = pd.DataFrame(
    validate(profile_type, profile)
    for profile_type, profile in validation_cases.items()
)
display(validation_summary_df)

,Profile Type,Age,Purposes Before,Purposes After,Purposes Removed,Start Times After,Modes After,Modes Removed,All Checks Passed
0,"Young, no household car",5-11,14,12,"Business, Commuting",24,8,"Car or van driver, Car or van passenger, Motor...",True
1,"Adult, work from home, with car",23-34,14,13,Commuting,24,11,None,True
2,"Adult, workplace, no household car",23-34,14,14,None,24,9,"Car or van driver, Car or van passenger",True
3,"Adult, workplace, with car",23-34,14,14,None,24,11,None,True


## 10. Inspect an Integrated Behavioural Representation

The integration was validated across four demographic cases. The young
profile without household car availability retained 12 trip purposes and
8 travel modes. The adult work-from-home profile retained 13 trip purposes
and all 11 travel modes. The workplace-based adult without household car
availability retained all 14 trip purposes and 9 travel modes. The
workplace-based control profile with household car availability retained
all 14 trip purposes and all 11 travel modes.

In [40]:
example_purpose = "Education"

example_behaviour = (
    validation_representation[
        "behavioural_representation"
    ]
)

example_purpose_distribution = (
    example_behaviour[
        "trip_purpose_distribution"
    ]
)

example_purpose_profile = (
    example_behaviour[
        "purpose_conditioned_distribution"
    ][example_purpose]
)


print("Demographic Profile")

display(
    pd.DataFrame(
        validation_representation[
            "demographic_profile"
        ].items(),
        columns=[
            "Attribute",
            "Profile Value",
        ],
    )
)


print("\nTrip-Purpose Distribution")

trip_purpose_table = pd.DataFrame([
    {
        "Trip Purpose": purpose,
        "Probability": probability,
    }
    for purpose, probability
    in example_purpose_distribution.items()
]).sort_values(
    "Probability",
    ascending=False,
)

display(
    trip_purpose_table.reset_index(
        drop=True
    )
)


print(
    f"\nTemporal Distribution: "
    f"{example_purpose}"
)

temporal_table = pd.DataFrame([
    {
        "Time Interval": interval,
        "Probability": probability,
    }
    for interval, probability
    in example_purpose_profile[
        "temporal_distribution"
    ].items()
])

display(temporal_table)


print(
    f"\nTravel-Mode Distribution: "
    f"{example_purpose}"
)

travel_mode_table = pd.DataFrame([
    {
        "Travel Mode": mode,
        "Probability": probability,
    }
    for mode, probability
    in example_purpose_profile[
        "travel_mode_distribution"
    ].items()
]).sort_values(
    "Probability",
    ascending=False,
)

display(
    travel_mode_table.reset_index(
        drop=True
    )
)


print(
    "\nTrip-purpose sum:",
    sum(
        example_purpose_distribution.values()
    ),
)

print(
    "Temporal sum:",
    sum(
        example_purpose_profile[
            "temporal_distribution"
        ].values()
    ),
)

print(
    "Travel-mode sum:",
    sum(
        example_purpose_profile[
            "travel_mode_distribution"
        ].values()
    ),
)

Demographic Profile


,Attribute,Profile Value
0,Age,5-11
1,Gender,Female
2,Household Type,Couple household: With dependent children
3,Economic Activity Status,Does not apply
4,Industry,Does not apply
5,Ethnic Group,White: Other White
6,Distance travelled to work,Does not apply
7,Car Ownership,No cars or vans in household



Trip-Purpose Distribution


,Trip Purpose,Probability
0,Education,0.328955
1,Other Escort,0.142932
2,Visiting Friends at Private Home,0.106983
3,Entertainment or Public Activity,0.078524
4,Personal Business,0.075901
5,Just Walk,0.053895
6,Escort Education,0.044314
7,Visiting Friends Elsewhere,0.043814
8,Day Trip,0.041310
9,Sport: Participate,0.036136



Temporal Distribution: Education


,Time Interval,Probability
0,0000 to 0059 (%),0.000000
1,0100 to 0159 (%),0.000000
2,0200 to 0259 (%),0.000000
3,0300 to 0359 (%),0.000306
4,0400 to 0459 (%),0.000044
5,0500 to 0559 (%),0.000112
6,0600 to 0659 (%),0.002540
7,0700 to 0759 (%),0.097998
8,0800 to 0859 (%),0.377559
9,0900 to 0959 (%),0.017586



Travel-Mode Distribution: Education


,Travel Mode,Probability
0,Walk,0.757551
1,Other local bus,0.125771
2,Other Private Transport,0.048151
3,Pedal Cycle,0.021939
4,Surface Rail,0.021709
5,Other Public Transport,0.013316
6,Taxi or minicab,0.011564
7,Non-local bus,0.000000



Trip-purpose sum: 1.0
Temporal sum: 1.0
Travel-mode sum: 1.0


## Summary

This notebook successfully integrated the constrained representative demographic profile space with the behavioural evidence derived from NTS0611a, NTS0502b, and NTS0409a.

For each feasible demographic profile, trip-purpose probabilities were retrieved using age and gender before demographic constraints were applied. Purpose-conditioned temporal distributions were then retained for every remaining trip purpose, while travel-mode distributions were adjusted using age and household car availability.

The integration was validated across four demographic cases. The young profile without household car availability retained 12 trip purposes and 8 travel modes. The adult work-from-home profile retained 13 trip purposes and all 11 travel modes. The workplace-based adult without household car availability retained all 14 trip purposes and 9 travel modes. The workplace-based control profile with household car availability retained all 14 trip purposes and all 11 travel modes.

All trip-purpose, temporal, and travel-mode probability distributions summed to one following constraint application. Alternatives unaffected by the relevant constraints retained their original probability distributions.

The resulting demographic–behavioural representation preserves complete conditional probability distributions and does not sample realised trip purposes, start times, or travel modes. It provides the behavioural input for the subsequent behaviour-to-POI integration stage and does not yet represent a final destination or complete human mobility representation.